# Experiment 3: ACS 2023 target and ACS 2014 auxiliary shadows
Train the target on 50,000 California 2023 records with expanded predictors.
Train 16 diverse shadows exclusively on California 2014 auxiliary records.
Each model fits its own preprocessing on its training partition only.
The label is nominal personal income >$50,000 in each year; inflation and survey
year shifts are part of this transfer experiment. Occupation and other category
codes remain native to each year; no cross-year code equivalence is assumed.
Income-derived predictors, identifiers, survey weights and allocation flags are
excluded. MIA uses logit, blind walk, entropy and cross-entropy.
Execution was authorized for this run. Ask before future cell/experiment runs.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import load_acs_income, split_acs_data, subset_target_training
from src.expanded_data import (
    load_expanded_acs, preprocess_expanded_partitions, expanded_feature_inventory,
    reference_split_config, EXPANDED_FEATURES, EXCLUDED, load_expanded_year,
)
from src.training import train_target_model, evaluate_model
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (collect_shadow_paper_features, collect_target_paper_features,
                                       make_probability_oracle)

torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 50_000
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED, 'Use the reference seed consistently.'
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=SEED)
SHADOW_RECORDS_PER_CLASS = 2000
N_SHADOW_MODELS = 16
# Native commuting units; all other added codes are categorical in the walk.
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
print('Training size:', TRAIN_SIZE, '| Input fields:', len(EXPANDED_FEATURES))
print('Reference split configuration:', SPLIT_CONFIG)


Training size: 50000 | Input fields: 48
Reference split configuration: {'validation_size': 10000, 'test_size': 10000, 'mia_nonmember_size': 10000, 'mia_member_size': 10000, 'seed': 42}


## Load 2023 target records and make stratified partitions
Split sizes and seed follow Experiment 2, applied to the new 2023 records.


In [2]:
expanded_source = load_expanded_year(2023)
reference_partitions = split_acs_data(expanded_source, **SPLIT_CONFIG)
expanded_base_partitions = reference_partitions
expanded_partitions = subset_target_training(
    expanded_base_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(expanded_partitions[name].index, reference_partitions[name].index)
assert set(expanded_partitions['mia_members'].index).issubset(expanded_partitions['train'].index)
for name in ['validation', 'test', 'mia_nonmembers']:
    assert set(expanded_partitions[name].index).isdisjoint(expanded_partitions['train'].index)
print('Eligible records:', len(expanded_source))
print('Unused records:', len(expanded_source) - TRAIN_SIZE
      - sum(len(expanded_partitions[name]) for name in ['validation', 'test', 'mia_nonmembers']))
display(pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
    for name, part in expanded_partitions.items()}).T)
feature_inventory = expanded_feature_inventory(expanded_partitions['train'])
display(feature_inventory)


Eligible records: 203278
Unused records: 123278


,records,positive_rate
validation,10000.0,0.49840
test,10000.0,0.49840
mia_nonmembers,10000.0,0.49840
train,50000.0,0.49844
mia_members,10000.0,0.49840


,group,type,unique_nonmissing,missing_fraction
feature,,,,
AGEP,original_ACSIncome,numeric,75,0.00000
COW,original_ACSIncome,categorical,8,0.00000
SCHL,original_ACSIncome,categorical,24,0.00000
MAR,original_ACSIncome,categorical,5,0.00000
OCCP,original_ACSIncome,categorical,521,0.00000
POBP,original_ACSIncome,categorical,214,0.00000
WKHP,original_ACSIncome,numeric,94,0.00000
SEX,original_ACSIncome,categorical,2,0.00000
RAC1P,original_ACSIncome,categorical,9,0.00000


## Preprocess using training records only
The output is sparse; the MLP densifies only the current batch. Cardinality
counts above refer to the original fields, not one-hot dimensions.


In [3]:
expanded_preprocessor, expanded_arrays = preprocess_expanded_partitions(expanded_partitions)
print('Encoded input dimensions:', expanded_arrays['train']['X'].shape[1])
print('Training CSR storage (MiB):', round(sum(
    value.nbytes for value in [expanded_arrays['train']['X'].data,
    expanded_arrays['train']['X'].indices, expanded_arrays['train']['X'].indptr]) / 2**20, 2))
pd.DataFrame({name: {'rows': values['X'].shape[0], 'encoded_features': values['X'].shape[1]}
    for name, values in expanded_arrays.items()}).T


Encoded input dimensions: 2628
Training CSR storage (MiB): 19.26


,rows,encoded_features
validation,10000,2628
test,10000,2628
mia_nonmembers,10000,2628
train,50000,2628
mia_members,10000,2628


## Train the target
Fixed 100 epochs, final weights retained. Validation is monitored but does not
select an epoch in this deliberately overfit experimental condition.


In [4]:
target_started = time.monotonic()
expanded_model, expanded_history, expanded_metadata = train_target_model(
    expanded_arrays['train']['X'], expanded_arrays['train']['y'],
    expanded_arrays['validation']['X'], expanded_arrays['validation']['y'], **TRAIN_CONFIG)
target_training_seconds = time.monotonic() - target_started
print('Target training seconds:', round(target_training_seconds, 1))
display(pd.DataFrame(expanded_history).iloc[::10])


Epoch 001: train loss=0.4282, validation loss=0.4267, validation accuracy=0.8000


Epoch 002: train loss=0.3797, validation loss=0.4050, validation accuracy=0.8155


Epoch 003: train loss=0.3527, validation loss=0.4096, validation accuracy=0.8139


Epoch 004: train loss=0.3154, validation loss=0.4284, validation accuracy=0.8092


Epoch 005: train loss=0.2627, validation loss=0.4863, validation accuracy=0.8059


Epoch 006: train loss=0.1970, validation loss=0.5726, validation accuracy=0.7944


... (91 outputs left out)

Epoch 098: train loss=0.0047, validation loss=1.8734, validation accuracy=0.7859


Epoch 099: train loss=0.0029, validation loss=1.9022, validation accuracy=0.7828


Epoch 100: train loss=0.0023, validation loss=2.0515, validation accuracy=0.7823
Target training seconds: 436.6


,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.428244,0.8000,0.895558,0.426668,0.82282,0.913921,0.398095
10,11,0.039398,0.7782,0.852767,1.208876,0.98990,0.999264,0.032519
20,21,0.014201,0.7799,0.858042,1.698819,0.99744,0.999964,0.007681
30,31,0.008899,0.7870,0.866741,1.719421,0.99718,0.999968,0.008481
40,41,0.007934,0.7825,0.861268,1.666093,0.99746,0.999955,0.007998
50,51,0.005735,0.7845,0.864147,1.771258,0.99968,1.000000,0.001150
60,61,0.003355,0.7834,0.862175,1.968391,0.99860,0.999998,0.003863
70,71,0.004763,0.7840,0.862311,1.828025,0.99964,0.999999,0.001190
80,81,0.005139,0.7807,0.857022,1.930711,0.99936,0.999998,0.002073
90,91,0.004362,0.7840,0.864252,1.793784,0.99980,1.000000,0.000669


## Income-prediction evaluation
These are target income metrics, not membership metrics. The test set was not
used for model selection. Do not choose target settings using these results.


In [5]:
expanded_metrics = {name: evaluate_model(expanded_model, expanded_arrays[name]['X'],
    expanded_arrays[name]['y']) for name in ['train', 'validation', 'test']}
display(pd.DataFrame(expanded_metrics).T)
print('Train-test accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['test']['accuracy']))


,accuracy,roc_auc,log_loss
train,0.99936,0.999999,0.001526
validation,0.78230,0.859698,2.051451
test,0.78890,0.865948,1.982785


Train-test accuracy gap (percentage points): 21.046


## 16 diverse expanded ACS 2014 shadow models
Diverse algorithms/settings and 40/20/40 splits follow the
restored Experiment 2 attack. Each preprocessor fits its own training records.
These shadows do not assume knowledge of the target architecture.
Cycle the five algorithms using unique seeds and splits. Models may reuse Adult
records across shadows; grouped attack splits keep repeated records together.


In [6]:
acs_auxiliary = load_expanded_year(2014)
assert set(acs_auxiliary.index).isdisjoint(expanded_source.index)
shadow_results = train_shadow_models(acs_auxiliary, seed=SEED, n_models=N_SHADOW_MODELS, schema='expanded_acs')
# Verify native-input queries reproduce probabilities computed during training.
for name, result in shadow_results.items():
    audit_rows = result['attack_data'].iloc[:32]
    native = acs_auxiliary.loc[audit_rows['record_id'], EXPANDED_FEATURES]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='acs')
    np.testing.assert_allclose(query(native),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
        rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch for {name}')
print('Probability oracle consistency verified for all 16 shadows.')
display(summarize_shadow_models(shadow_results))


Training shadow 1/16: xgboost


Training shadow 2/16: random_forest


Training shadow 3/16: mlp


Epoch 001: train loss=0.3997, validation loss=0.3652, validation accuracy=0.8339


Epoch 002: train loss=0.3609, validation loss=0.3552, validation accuracy=0.8387


Epoch 003: train loss=0.3497, validation loss=0.3516, validation accuracy=0.8395


... (34 outputs left out)

Training shadow 15/16: logistic_regression


Training shadow 16/16: xgboost


Probability oracle consistency verified for all 16 shadows.


records  accuracy   roc_auc  log_loss
shadow_model           partition                                        
xgboost_01             train         73576  0.843332  0.918794  0.348485
                       validation    36789  0.831961  0.908915  0.365237
                       nonmembers    73576  0.833166  0.908590  0.365662
random_forest_02       train         73576  0.895156  0.963715  0.305829
                       validation    36789  0.826878  0.901655  0.392022
                       nonmembers    73576  0.824372  0.900115  0.393453
mlp_03                 train         73576  0.854749  0.926123  0.329771
                       validation    36789  0.839544  0.915249  0.351638
                       nonmembers    73576  0.840260  0.914639  0.352614
decision_tree_04       train         73576  0.857182  0.931300  0.316238
                       validation    36789  0.803365  0.855743  1.151549
                       nonmembers    73576  0.798453  0.854496  1.132455
logistic_regression_05 train         73576  0.848796  0.920043  0.342480
                       validation    36789  0.835685  0.906960  0.369138
                       nonmembers    73576  0.832676  0.905313  0.371709
xgboost_06             train         73576  0.843645  0.918923  0.348288
                       validation    36789  0.833592  0.909114  0.364757
                       nonmembers    73576  0.832255  0.908488  0.365835
random_forest_07       train         73576  0.893851  0.963988  0.304787
                       validation    36789  0.824350  0.900185  0.392477
                       nonmembers    73576  0.821137  0.897831  0.396558
mlp_08                 train         73576  0.873097  0.941752  0.295663
                       validation    36789  0.837696  0.912051  0.358834
                       nonmembers    73576  0.837610  0.912310  0.358417
decision_tree_09       train         73576  0.855428  0.929687  0.319742
                       validation    36789  0.799831  0.852819  1.176528
                       nonmembers    73576  0.800071  0.854773  1.136423
logistic_regression_10 train         73576  0.847736  0.919022  0.345006
                       validation    36789  0.832586  0.904105  0.373610
                       nonmembers    73576  0.834443  0.908028  0.366182
xgboost_11             train         73576  0.842544  0.918065  0.350210
                       validation    36789  0.831553  0.908831  0.365171
                       nonmembers    73576  0.834783  0.909777  0.363727
random_forest_12       train         73576  0.895102  0.964055  0.305577
                       validation    36789  0.826198  0.900214  0.393682
                       nonmembers    73576  0.825854  0.901091  0.392064
mlp_13                 train         73576  0.857494  0.929113  0.324575
                       validation    36789  0.837370  0.914066  0.353423
                       nonmembers    73576  0.838439  0.912665  0.356493
decision_tree_14       train         73576  0.854790  0.930075  0.318332
                       validation    36789  0.796189  0.855388  1.090280
                       nonmembers    73576  0.800125  0.858216  1.082414
logistic_regression_15 train         73576  0.848075  0.919368  0.343927
                       validation    36789  0.832477  0.905450  0.371749
                       nonmembers    73576  0.834185  0.906840  0.368393
xgboost_16             train         73576  0.843957  0.919106  0.348214
                       validation    36789  0.832042  0.908496  0.366079
                       nonmembers    73576  0.833410  0.908997  0.365118

## Extract shadow features and fit the random forest attack
Features: class-scaled logit, blind walk, entropy and cross-entropy. Use 2,000
members and 2,000 non-members per shadow. Attack record-ID grouping prevents
repeated 2014 records from crossing attack partitions. Select both thresholds
on auxiliary validation only; no target membership labels train this classifier.


In [7]:
shadow_attack_data, shadow_query_diagnostics = collect_shadow_paper_features(
    shadow_results, acs_auxiliary, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=SEED)
display(pd.DataFrame(shadow_query_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(attack_result['metrics']).T)


Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (120 outputs left out)

Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
xgboost_01,124244,164000,0.607375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_02,127052,164000,0.6455,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_03,125358,164000,0.61925,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_04,103051,164000,0.4275,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_05,121050,164000,0.56875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_06,124575,164000,0.609,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_07,127145,164000,0.645875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_08,123703,164000,0.593,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_09,104318,164000,0.434625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_10,121841,164000,0.579625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,38419.0,19317.0,0.569328,0.567673,0.645868,0.039654,0.485614,5194.0,13908.0,2638.0,16679.0,0.591197,0.018999,0.004397
validation,12849.0,6376.0,0.513736,0.515967,0.517024,0.011449,0.485614,1427.0,5046.0,1202.0,5174.0,0.591197,0.011449,0.009733
test,12732.0,6307.0,0.510603,0.513391,0.514880,0.008720,0.485614,1366.0,5059.0,1172.0,5135.0,0.591197,0.010623,0.012607


## Final MIA evaluation on the expanded target
Query the expanded native input schema, preserving categorical codes and
handling numeric commute fields in native units. The walk still generates
field-valid but potentially implausible combinations, and its dimensionality
matches the expanded auxiliary schema. Thresholds are frozen from auxiliary validation.


In [8]:
target_attack_data, target_query_diagnostics = collect_target_paper_features(
    expanded_model, expanded_preprocessor, expanded_partitions,
    config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
display(pd.Series(target_query_diagnostics, name='expanded target query diagnostics'))
target_mia_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
target_mia_metrics['tpr_at_accuracy_threshold'] = target_mia_metrics['tp'] / target_mia_metrics['members']
target_mia_metrics['fpr_at_accuracy_threshold'] = target_mia_metrics['fp'] / (
    target_mia_metrics['records'] - target_mia_metrics['members'])
display(pd.DataFrame({'2014 auxiliary attack test': attack_result['metrics']['test'],
                      'Expanded ACS target MIA': target_mia_metrics}).T)


Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


Attack feature extraction: 3,072/20,000


... (32 outputs left out)

Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


record_queries                                                       535043
max_record_queries                                                   820000
walk_censored_fraction                                               0.4509
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: expanded target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold
2014 auxiliary attack test,12732.0,6307.0,0.510603,0.513391,0.514880,0.00872,0.485614,1366.0,5059.0,1172.0,5135.0,0.591197,0.010623,0.012607,NaN,NaN
Expanded ACS target MIA,20000.0,10000.0,0.607750,0.607750,0.642688,0.01510,0.485614,2665.0,7335.0,510.0,9490.0,0.591197,0.056000,0.029000,0.949,0.7335


## Small reproducibility artifacts
Save only summaries, feature inventory and row assignments (no model checkpoint).
Source CSVs and the other experiment notebooks remain unchanged.


In [9]:
result_dir = Path('artifacts/exp_3_2023_2014')
result_dir.mkdir(parents=True, exist_ok=True)
feature_inventory.to_csv(result_dir / 'feature_inventory.csv')
for name, part in expanded_partitions.items():
    pd.Series(part.index, name='record_id').to_csv(result_dir / f'{name}_ids.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(result_dir / 'shadow_metrics.csv')
summary = dict(execution_status='all_cells_completed',
    attack_metrics_status='oracle_consistency_verified', target_year=2023, auxiliary_year=2014, n_shadow_models=N_SHADOW_MODELS,
    shadow_seeds={name: result['seed'] for name, result in shadow_results.items()},
    original_feature_count=len(EXPANDED_FEATURES),
    encoded_feature_count=expanded_arrays['train']['X'].shape[1],
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, training=expanded_metadata,
    target_training_seconds=target_training_seconds, target_metrics=expanded_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_mia_metrics=target_mia_metrics,
    target_query_diagnostics=target_query_diagnostics)
(result_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Summary and exact row IDs written to', result_dir)


Summary and exact row IDs written to artifacts\exp_3_2023_2014
